### Exploring VCF Storage Solutions
# Notebook 5.1. OpenCGA Part I: Initiation and general operations

2025-03-20 Daniel P. Brink

# Summary

This the first of the notebooks in this project that focuses on OpenCGA. 

**NOTE!** The investigation in this notebook was made using an old version of OpenCGA (v2.2.1) from 2021 since that was the only version that I was able to get running during the testing. Therefore, the findings in this notebook only represent that version. It is possible that some of the errors encountered here have been addressed in later versions of OpenCGA. Please keep that in mind when reading.

The notebook covers: 

- Overview of the tool (Section 1)
- Installation (Section 2)
- Basic data ingestion (Section 3)
- Commands for performing basic queries and subsets (Section 4)
- Export of the dataset from MongoDB back to VCF (Section 5)

Key findings:

- OpenCGA uses a NoSQL database to store and interact with the data from the VCF files. This is a markedly different strategy from the sparse arrays used by TileDB-VCF and VCF Zarr.

- This tool is data management-focused system, which sets it apart from the data storage-focused strategies of TileDB-VCF and VCF Zarr. To use OpenCGA, an admin account need to create a user account, which in turn is used to log in to the system and create projects. Ingested data is assigned to a project.

- Another thing that sets it apart from the other two tools is that OpenCGA is designed to be deployed as a REST server. That can be beneficial from both a user and an admin perspective since that design enforces "single source of truth" data for all users that are given access to a project. In this notebook the server will be run as localhost for the sake of testing. 

- The documentation is poor and incomplete. It required a lot of detective work to get a working installation running, and in the end that was based on running an old Docker image that happened to have all dependencies and scripts needed to understand how to run the software.

- The OpenCGA GitHub repository is continusly updated by the company Zetta Genomics. They offer a commercial solution called XetaBase that seem to be based on OpenCGA. Given the state of the documentation and how the issues section of the GitHub repo has been closed, it seems that the OpenCGA developers have pivoted towards their commercial offering.

- Big operations such as data ingestion and export are run as jobs. Queries are made with the OpenCGA CLI tools, but should technically be possible to do directly on the MongoDB database if one is willing to put in the time needed to understand the database schema.

- The export from OpenCGA back to VCF resulted in an error. No intermediate file was recovered.

# 1. Introduction

OpenCGA is a storage system for genetic variants. The [version that will be investigated here is open-source](https://github.com/opencb/opencga). At the time of testing, the repository is maintained by the company Zetta Genomics, that also offer a commercial version called [Xetabase](https://zettagenomics.com/xetabase/). The OpenCGA project seem to have been around since [at least 2014](https://github.com/opencb/opencga/releases/tag/0.3.2) and is still seing active releases on GitHub. There is also a related open source project called [IVA](https://github.com/opencb/iva) that offer a GUI for OpenCGA, but investigating it was outside the scope of this notebook. Interestingly, it is not very clear from the documentation what GCA stands for in OpenCGA. None of the docs, webpages, and repos seem to spell it out. After some digging, this [slide deck from 2019](http://courseswiki.clinbioinfosspa.es/_media/gatk2019/introduction_to_opencga_and_iva.pdf) mentions on page 38 that CGA stands for Computational Genomics Analysis. 

The big case-study for this software seem to be [the 100,000 genomes project](https://github.com/opencb/opencga/blob/develop/docs/case-studies/genomics-england-research.md) at [Genomics England](https://www.genomicsengland.co.uk/news/mongodb-used-to-power-data-science-for-100k-project). The European Variant Archive (EVA) are sometimes mentioned as a big player that are or have been using OpenCGA for their services. Looking through the EVA GitHub repositories, it at least seems like [one of their pipelines uses OpenCGA](https://github.com/EBIvariation/eva-pipeline). If that pipeline is still being used in production at EVA remains to be investigated, but the latest GitHub release of the pipeline is from 2018.

## 1.1. What tools are needed to use OpenCGA?

Something that needs to be said upfront is that the documenation for OpenCGA is sparse at best. The little docs that exist seem to describe old versions and often contain examples that no longer apply. The README in the GitHub repo states that the documentation is on their GitHub wiki [here](https://github.com/opencb/opencga/wiki/). That Wiki opens with a message saying that the docs have been moved elsewhere, but the link is broken. A Google search revealed [this documentation page](https://docs.opencga.opencb.org/) which seem to refer to version 2.2. At the time of this investigation, the latest OpenCGA release is v3.4. The v2.2 docs contain many empty pages and some broken links (but some of pages can be found by looking through the [GH repo](https://github.com/opencb/opencga/tree/develop/docs)).

These dependencies were used in for the tests in this notebook:

- Docker
- OpenCGA
- MongoDB
- Apache SOLR

## 1.2. Resources that aided and inspired various aspects of this notebook

- The `opencga-demo` Docker [image](https://hub.docker.com/r/opencb/opencga-demo). Especially the `init.sh` file contained within the container.

- The aforementioned documentation pages [here](https://github.com/opencb/opencga/wiki/) and [here](https://docs.opencga.opencb.org/).


# 2. Installation

Several different installations were tried to attempt to get OpenCGA to work. It was difficult to figure our all dependencies that were needed to be able to use the tool without errors, as the little documentation that exists was either outdated or incomplete. In the end, the only successul strategy was to use a 2021 Docker image for a [demo setup that came with every dependency pre-configured](https://hub.docker.com/r/opencb/opencga-demo). Specifically, the `opencb/opencga-demo:2.2.1-SNAPSHOT` image was used, containing OpenCGA 2.2.1. At the time of writing, this Docker image is four year old. Therefore, the investigation in this notebook does not reflect any of the [code updates done since](https://github.com/opencb/opencga/releases). Please keep this in mind while reading! It is likely that some of the obstacles encountered during this notebook have been adressed in later versions of OpenCGA.

[Docker desktop](https://www.docker.com/products/docker-desktop/) was installed on a Mac M3 laptop and was running in the background during all the tests in this notebook. To download the latest version of the image, the following command was run in the terminal:

```
docker pull opencb/opencga-demo:2.2.1-SNAPSHOT
```

There is a short instruction on the DockerHub page saying to run the image using `--env load=true`. This sets up a user profile and loads some test data. We will run the command to see what we can learn about the process, but will eventually try to setup a user and ingest our own example data. Because of this, let's name the container that we will spin up from the image as `opencga-demo_example_data`.

In [1]:
%%time
!docker run -d --name opencga-demo_example_data --env load=true -p 9090:9090 opencb/opencga-demo:2.2.1-SNAPSHOT

3e2d5f5f1fe2fb7824b15e645b1ac7afb5042dec5a2643e07f45fc72955cc6f1
CPU times: user 19.4 ms, sys: 29.4 ms, total: 48.8 ms
Wall time: 532 ms


(The warning message about the mismatch between the image and the host platform can be disregarded. Everything seemed to run fine, and the error messages that were encountered did not seem to be related to CPU architecture.)

Upon starting the container, an initiation script named `init.sh` will run, as is evident by:

In [2]:
%%time
!docker inspect opencga-demo_example_data | head

[
    {
        "Id": "3e2d5f5f1fe2fb7824b15e645b1ac7afb5042dec5a2643e07f45fc72955cc6f1",
        "Created": "2025-03-10T13:45:08.686496764Z",
        "Path": "/bin/bash",
        "Args": [
            "-c",
            "/opt/opencga/demo/init.sh"
        ],
        "State": {
CPU times: user 5.91 ms, sys: 19 ms, total: 24.9 ms
Wall time: 284 ms


If we inspect the file with e.g.
```
!docker exec -it opencga-demo_example_data less "/opt/opencga/demo/init.sh"
```
we can learn all the operations that are run in the container to initate the the OpenCGA instance. For the sake of brevity, the output of the full file is not presented here. 

In short, what happens is:

- MongoDB is started (a database engine)
- Apache SOLR is started (a search engine)
- an OpenCGA catalog (for user and data management) is installed with `opencga-admin.sh catalog install`
- if `--env load=true` the `/opt/opencga/demo/load-demo.sh` the helper script is excecuted
- a (localhost) server is started with `opencga-admin.sh server rest --start`
- a daemon is started that listens to the server and prints output to the Docker container logs: `./opencga-admin.sh catalog daemon --start`

Running the container with `--env load=true` threw an error in the data ingestion job during the step related to Variant Annotation. It seemed to be a matter of a missing dependency or database. Despite this, it turns out that everything else works well enough for us to try to work with the container with our own data and user accounts.

Some operations in OpenCGA are run as jobs and will thus not print their output to the Jupyter cells. For cases when the content of the log is relevant for this notebook, it will be copied from the Docker container log to the markdown cells.

# 3. Ingesting VCF data

The workflow in this section takes a cue from the helper script `/opt/opencga/demo/load-demo.sh` found in the container.

First, we will need to create a new user with `opencga-admin.sh`.
Then will will log in to the system using the credentials for the new user. This, and all subsequent operations, will be done with `opencga.sh`.

## 3.1. Create a user, log in, and create a project

We start by creating a new user called `testuser`. Running the command is the following manner will create the user in the `organization` called `opencga`. For now, we can be fine with that, but in a real-life project it would probably be of interest to control the user's organization in more detail.

(An admin password is needed to run `./opencga-admin.sh` in this container. It is available from `init.sh`.)

The following creates a user using the minimum mandatory flags for this version of `opencga`:

```
!docker exec -it opencga-demo_example_data ./opencga-admin.sh users create -u testuser --email test@example.com --name "test user" --user-password 1234@User
```

If the admin password was sucessfully input, this message should hopefully display: `The user has been successfully created`.

We can now log in as the user. The rest of the commands will be done through `./opencga.sh` and we will no longer need to use `./opencga-admin.sh` or the admin password.

In [3]:
%%time
!docker exec -it opencga-demo_example_data ./opencga.sh users login -u testuser -p 1234@User

You have been logged in correctly: testuser

What's next:
    Try Docker Debug for seamless, persistent debugging tools in any container or image → docker debug opencga-demo_example_data
    Learn more at https://docs.docker.com/go/debug-cli/
CPU times: user 83.5 ms, sys: 42.7 ms, total: 126 ms
Wall time: 3.72 s


(There is a default session timer for how long a user can be logged in. If suddenly any messages related to permissions appear, chances are that you just need to log in again.)


Now we can start doing some data management. We can create a project for our data with the following commands:

(Dummy strings are used for the mandatory strings. For simplicity, we can set `name` and `id` to the same string, `testproj`. Later on, we will be calling on the project `id` quite a bit)

In [4]:
%%time
!docker exec -it opencga-demo_example_data ./opencga.sh projects create --name testproj --organism-assembly test_organism_assembly --id testproj --organism-scientific-name test_organism_scientific_name

1

What's next:
    Try Docker Debug for seamless, persistent debugging tools in any container or image → docker debug opencga-demo_example_data
    Learn more at https://docs.docker.com/go/debug-cli/
CPU times: user 73.4 ms, sys: 39.4 ms, total: 113 ms
Wall time: 3.62 s


The output of the previous cell does not tell much, realy, but we can check to see that the project has been created for our user by running:

In [5]:
%%time
!docker exec -it opencga-demo_example_data ./opencga.sh users projects --user testuser

#ID	NAME	ORGANISM	ASSEMBLY	DESCRIPTION	#STUDIES	STATUS
testproj	testproj	test_organism_scientific_name	test_organism_assembly	-	0	READY

What's next:
    Try Docker Debug for seamless, persistent debugging tools in any container or image → docker debug opencga-demo_example_data
    Learn more at https://docs.docker.com/go/debug-cli/
CPU times: user 73.4 ms, sys: 29.8 ms, total: 103 ms
Wall time: 3.94 s


We can further create a `study` within the project:

In [6]:
%%time
!docker exec -it opencga-demo_example_data ./opencga.sh studies create --name teststudy --id teststudy --project testproj

#ID	NAME	DESCRIPTION	#GROUPS	SIZE	#FILES	#SAMPLES	#COHORTS	#INDIVIDUALS	#JOBS	#VARIABLE_SETS	STATUS
teststudy	teststudy	-	2	0	0	0	0	0	0	0	READY

What's next:
    Try Docker Debug for seamless, persistent debugging tools in any container or image → docker debug opencga-demo_example_data
    Learn more at https://docs.docker.com/go/debug-cli/
CPU times: user 81.8 ms, sys: 34.3 ms, total: 116 ms
Wall time: 4.05 s


It is a little unclear if a study is needed for a project to be able to accept data, but that was how it was done in the example script at `/opt/opencga/demo/load-demo.sh` in the container.

## 3.2. Ingesting the data

To ingest data, we first need to define the path within the catalog in which the data will be stored.

In [7]:
%%time
!docker exec -it opencga-demo_example_data ./opencga.sh files create --study 'testuser@testproj:teststudy' --path 'data' --type 'DIRECTORY'

#ID	NAME	TYPE	FORMAT	BIOFORMAT	DESCRIPTION	CATALOG_PATH	FILE_SYSTEM_URI	STATUS	SIZE	INDEX_STATUS	RELATED_FILES	SAMPLES
data:	data	DIRECTORY	NONE	NONE	-	data/	file:///opt/opencga/sessions/users/testuser/projects/31/43/data/	READY	0	NONE	-	-

What's next:
    Try Docker Debug for seamless, persistent debugging tools in any container or image → docker debug opencga-demo_example_data
    Learn more at https://docs.docker.com/go/debug-cli/
CPU times: user 86 ms, sys: 36.1 ms, total: 122 ms
Wall time: 3.93 s


To ingest the data, we can first copy the example data from the local disk to the Docker container:

In [8]:
%%time
!docker cp input_data_temp/1kG_p3_chr1_first_200_samples_c1.vcf.gz opencga-demo_example_data:/opt/opencga/misc/1kG_p3_chr1_first_200_samples_c1.vcf.gz

Successfully copied 83.5MB to opencga-demo_example_data:/opt/opencga/misc/1kG_p3_chr1_first_200_samples_c1.vcf.gz
CPU times: user 6.62 ms, sys: 13.1 ms, total: 19.7 ms
Wall time: 556 ms


Looks good. As a sanity-check, we can verify that the file is present inside the container:

In [9]:
%%time
!docker exec -it opencga-demo_example_data ls -l /opt/opencga/misc/1kG_p3_chr1_first_200_samples_c1.vcf.gz

-rw-r--r--    1 501      dialout   83492756 Feb  5 14:25 /opt/opencga/misc/1kG_p3_chr1_first_200_samples_c1.vcf.gz

What's next:
    Try Docker Debug for seamless, persistent debugging tools in any container or image → docker debug opencga-demo_example_data
    Learn more at https://docs.docker.com/go/debug-cli/
CPU times: user 9.48 ms, sys: 15.2 ms, total: 24.7 ms
Wall time: 366 ms


We can now create a link to the files so that `opencga` knows where to access the data:

In [10]:
%%time
!docker exec -it opencga-demo_example_data ./opencga.sh files link --study teststudy --path data --input /opt/opencga/misc/1kG_p3_chr1_first_200_samples_c1.vcf.gz 

#ID	NAME	TYPE	FORMAT	BIOFORMAT	DESCRIPTION	CATALOG_PATH	FILE_SYSTEM_URI	STATUS	SIZE	INDEX_STATUS	RELATED_FILES	SAMPLES
data:1kG_p3_chr1_first_200_samples_c1.vcf.gz	1kG_p3_chr1_first_200_samples_c1.vcf.gz	FILE	VCF	VARIANT	-	data/1kG_p3_chr1_first_200_samples_c1.vcf.gz	file:///opt/opencga/misc/1kG_p3_chr1_first_200_samples_c1.vcf.gz	READY	83492756	NONE	-	HG00096,HG00097,HG00099,HG00100,HG00101,HG00102,HG00103,HG00105,HG00106,HG00107,HG00108,HG00109,HG00110,HG00111,HG00112,HG00113,HG00114,HG00115,HG00116,HG00117,HG00118,HG00119,HG00120,HG00121,HG00122,HG00123,HG00125,HG00126,HG00127,HG00128,HG00129,HG00130,HG00131,HG00132,HG00133,HG00136,HG00137,HG00138,HG00139,HG00140,HG00141,HG00142,HG00143,HG00145,HG00146,HG00148,HG00149,HG00150,HG00151,HG00154,HG00155,HG00157,HG00158,HG00159,HG00160,HG00171,HG00173,HG00174,HG00176,HG00177,HG00178,HG00179,HG00180,HG00181,HG00182,HG00183,HG00185,HG00186,HG00187,HG00188,HG00189,HG00190,HG00231,HG00232,HG00233,HG00234,HG00235,HG00236,HG00237,HG00238,HG002

Alright, it seems to have detected all 200 samples from the VCF file.

To ingest the data into MongoDB database for this project, we need to use `./opencga.sh operations variant-index`. The name of this command might be misleading to users that are used to `bcftools`. Here, _index_ is not referring to creating a .csi or tabix index of the VCF file, it is about indexing it in the MongoDB database.

In [11]:
%%time
!docker exec -it opencga-demo_example_data ./opencga.sh operations variant-index --file 1kG_p3_chr1_first_200_samples_c1.vcf.gz --job-id 'variant-index'

#ID	TOOL_ID	SUBMISSION_DATE	STATUS	EVENTS	START	RUNNING_TIME	INPUT	OUTPUT
variant-index	variant-index	2025-03-10 13:48:51	PENDING	-	-	-	1kG_p3_chr1_first_200_samples_c1.vcf.gz	-

What's next:
    Try Docker Debug for seamless, persistent debugging tools in any container or image → docker debug opencga-demo_example_data
    Learn more at https://docs.docker.com/go/debug-cli/
CPU times: user 86.4 ms, sys: 39.6 ms, total: 126 ms
Wall time: 3.69 s


It is possible to get the full log of the job by running:

```
!docker exec -it opencga-demo_example_data ./opencga.sh jobs log
```

The logs are extensive, though, and too large to print in this notebook. A list of the jobs can be found by running:

In [12]:
%%time
!docker exec -it opencga-demo_example_data ./opencga.sh jobs search

#ID	TOOL_ID	SUBMISSION_DATE	STATUS	EVENTS	START	RUNNING_TIME	INPUT	OUTPUT
variant-index	variant-index	2025-03-10 13:48:51	DONE	WARNING:1	2025-03-10 13:50:51	00:08:05	1kG_p3_chr1_first_200_samples_c1.vcf.gz	1kG_p3_chr1_first_200_samples_c1.vcf.gz.duplicated.tsv

What's next:
    Try Docker Debug for seamless, persistent debugging tools in any container or image → docker debug opencga-demo_example_data
    Learn more at https://docs.docker.com/go/debug-cli/
CPU times: user 105 ms, sys: 47.8 ms, total: 153 ms
Wall time: 4.29 s


The job finished sucessfully after about 8 minutes. The warning message mentioned in the job overview (see previous output of the previous cell) might be referring to the detection of a few duplicated variants. This message was printed to the logs just before the job finished:

```
[main] INFO  MongoDBVariantStoragePipeline:781 - ============================================================
[main] INFO  MongoDBVariantStoragePipeline:782 - Check loaded file '1kG_p3_chr1_first_200_samples_c1.vcf.gz' (1)
[main] ERROR MongoDBVariantStoragePipeline:797 - There were 4 duplicated variants not inserted. 
[main] INFO  MongoDBVariantStoragePipeline:814 - Final number of loaded variants: 1346514
[main] INFO  MongoDBVariantStoragePipeline:817 - ============================================================
```

In notebook 1 we ran `!bcftools view --no-header {downsampled_bcf} |wc -l` to count the number of lines in the example data (excluding headers). It was 1326846, but the number reported here is 1346514. From the message, it would have been reasonable to expect 3 or 4 fewer variants depending on how the omission process looked like, but here the number implies that ~20,000 extra variants were added! Could it be a matter of how INDELs or structural variants are treated? 

Understanding where these numbers come from will be essential for understanding how the system works and how it handles data. But for the scope of this notebook, perhaps we should not stare too much at the variant count and just accept that we are at least in the ballpark? We will accept this discrepency for now. 

# 4. Working with the data: queries, subsetting, filtering

## 4.1. How is the data organized and accessed in the MongoDB?

The data has now been ingested and is stored in the mongoDB database. This Docker image runs an older version of MongoDB in which the interactive CLI is launched with `mongo` and not with `mongosh` like in newer versions. To check the existing databases, we can do

```
docker exec -it opencga-demo_example_data mongo
show dbs
```

or, as a one-liner, use `mongo --eval`:

In [15]:
%%time
!docker exec -it opencga-demo_example_data mongo --eval "db.adminCommand('listDatabases').databases.forEach(function(db) { print('Name: ' + db.name + ', Size: ' + (db.sizeOnDisk / (1024 * 1024)).toFixed(2) + ' MiB'); })"

MongoDB shell version v4.0.5
connecting to: mongodb://127.0.0.1:27017/?gssapiServiceName=mongodb
Implicit session: session { "id" : UUID("1ed3b9db-0ef0-4417-ba70-18e8e0f103cd") }
MongoDB server version: 4.0.5
Name: admin, Size: 0.09 MiB
Name: config, Size: 0.18 MiB
Name: local, Size: 541.02 MiB
Name: opencga-demo_catalog, Size: 6.99 MiB
Name: opencga-demo_demo_family, Size: 262.76 MiB
Name: opencga-demo_testuser_testproj, Size: 689.03 MiB

What's next:
    Try Docker Debug for seamless, persistent debugging tools in any container or image → docker debug opencga-demo_example_data
    Learn more at https://docs.docker.com/go/debug-cli/
CPU times: user 18.9 ms, sys: 22.5 ms, total: 41.3 ms
Wall time: 762 ms


There are a few databases in here.  `opencga-demo_demo_family` was created because we ran the Docker image with `--env load=true`. The database we created is clearly `opencga-demo_testuser_testproj`. If we go by these numbers, the 80 MiB VCF now occupies 689 MiB as a database. Sure, the comparison might not be perfect since we do not know exactly how MongoDB `db.sizeOnDisk` calculates the disksize, but for whatever it is worth, it seems that we at least haven't compressed the data by ingesting it.

MongoDB is a document-oriented NoSQL database. To list all "collections", i.e. the related documents within the database (analogous to listing all related tables in a SQL database), we can use:

In [16]:
%%time
!docker exec -it opencga-demo_example_data mongo opencga-demo_testuser_testproj --eval "printjson(db.getCollectionNames())"

MongoDB shell version v4.0.5
connecting to: mongodb://127.0.0.1:27017/opencga-demo_testuser_testproj?gssapiServiceName=mongodb
Implicit session: session { "id" : UUID("7cc48766-d078-410e-975f-1e46fbcfa671") }
MongoDB server version: 4.0.5
[
	"cohorts",
	"files",
	"project",
	"samples",
	"stage_study_1",
	"studies",
	"tasks",
	"variants"
]

What's next:
    Try Docker Debug for seamless, persistent debugging tools in any container or image → docker debug opencga-demo_example_data
    Learn more at https://docs.docker.com/go/debug-cli/
CPU times: user 19.5 ms, sys: 21.2 ms, total: 40.6 ms
Wall time: 633 ms


Here we learn that the database we created contains not just the data. Let's poke a little at these different collections.

`files` seem to collect metadata that more or less align with the information from the VCF header.

In [17]:
%%time
!docker exec -it opencga-demo_example_data mongo opencga-demo_testuser_testproj --eval "db.files.find()"

MongoDB shell version v4.0.5
connecting to: mongodb://127.0.0.1:27017/opencga-demo_testuser_testproj?gssapiServiceName=mongodb
Implicit session: session { "id" : UUID("35065c95-c314-4f8a-886f-cc3564a18a5f") }
MongoDB server version: 4.0.5
{ "_id" : "1_1", "attributes" : {  }, "id" : 1, "name" : "1kG_p3_chr1_first_200_samples_c1.vcf.gz", "path" : "/opt/opencga/misc/1kG_p3_chr1_first_200_samples_c1.vcf.gz", "samples" : [ 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37, 38, 39, 40, 41, 42, 43, 44, 45, 46, 47, 48, 49, 50, 51, 52, 53, 54, 55, 56, 57, 58, 59, 60, 61, 62, 63, 64, 65, 66, 67, 68, 69, 70, 71, 72, 73, 74, 75, 76, 77, 78, 79, 80, 81, 82, 83, 84, 85, 86, 87, 88, 89, 90, 91, 92, 93, 94, 95, 96, 97, 98, 99, 100, 101, 102, 103, 104, 105, 106, 107, 108, 109, 110, 111, 112, 113, 114, 115, 116, 117, 118, 119, 120, 121, 122, 123, 124, 125, 126, 127, 128, 129, 130, 131, 132, 133, 134, 135, 136, 137, 

`project` is basically the information we entered when we created the project with `opencga.sh`:

In [18]:
%%time
!docker exec -it opencga-demo_example_data mongo opencga-demo_testuser_testproj --eval "db.project.find()"

MongoDB shell version v4.0.5
connecting to: mongodb://127.0.0.1:27017/opencga-demo_testuser_testproj?gssapiServiceName=mongodb
Implicit session: session { "id" : UUID("0127b66f-4e6f-4f31-a476-c48dbf1af0b8") }
MongoDB server version: 4.0.5
{ "_id" : "META", "_lock" : { "write" : null }, "annotation" : { "current" : null, "saved" : [ ] }, "assembly" : "test_organism_assembly", "attributes" : {  }, "release" : 1, "species" : "test_organism_scientific_name", "counters" : { "study" : 1, "file" : 1, "sample" : 200, "task" : 1, "cohort" : 1 } }

What's next:
    Try Docker Debug for seamless, persistent debugging tools in any container or image → docker debug opencga-demo_example_data
    Learn more at https://docs.docker.com/go/debug-cli/
CPU times: user 16.4 ms, sys: 20.6 ms, total: 37 ms
Wall time: 657 ms


`samples` contain a list of all the samples, as expected (here truncated to show only the first five)

In [21]:
%%time
!docker exec -it opencga-demo_example_data mongo opencga-demo_testuser_testproj --eval "db.samples.find().limit(5)"

MongoDB shell version v4.0.5
connecting to: mongodb://127.0.0.1:27017/opencga-demo_testuser_testproj?gssapiServiceName=mongodb
Implicit session: session { "id" : UUID("092d42fc-feb2-4660-a403-3c9271e9f803") }
MongoDB server version: 4.0.5
{ "_id" : "1_1", "attributes" : {  }, "cohorts" : [ 1 ], "father" : null, "files" : [ 1 ], "id" : 1, "mother" : null, "name" : "HG00096", "secondaryIndexCohorts" : [ ], "splitData" : null, "stats" : null, "status" : { "index" : "READY" }, "studyId" : 1, "_lock" : { "write" : null } }
{ "_id" : "1_2", "attributes" : {  }, "cohorts" : [ 1 ], "father" : null, "files" : [ 1 ], "id" : 2, "mother" : null, "name" : "HG00097", "secondaryIndexCohorts" : [ ], "splitData" : null, "stats" : null, "status" : { "index" : "READY" }, "studyId" : 1, "_lock" : { "write" : null } }
{ "_id" : "1_3", "attributes" : {  }, "cohorts" : [ 1 ], "father" : null, "files" : [ 1 ], "id" : 3, "mother" : null, "name" : "HG00099", "secondaryIndexCohorts" : [ ], "splitData" : null, "s

`stage_study_1` seem to contain the ALT and REF alleles for each variant?

In [24]:
%%time
!docker exec -it opencga-demo_example_data mongo opencga-demo_testuser_testproj --eval "db.stage_study_1.find()"

MongoDB shell version v4.0.5
connecting to: mongodb://127.0.0.1:27017/opencga-demo_testuser_testproj?gssapiServiceName=mongodb
Implicit session: session { "id" : UUID("5317e009-735a-40dd-abdb-4dfc0be03a7f") }
MongoDB server version: 4.0.5
{ "_id" : " 1:    173052:C:A", "1" : { "1" : null, "new" : false }, "_i" : [ "1" ], "alt" : "A", "end" : 173052, "ref" : "C" }
{ "_id" : " 1:    230105:G:C", "1" : { "1" : null, "new" : false }, "_i" : [ "1" ], "alt" : "C", "end" : 230105, "ref" : "G" }
{ "_id" : " 1:    232449:G:A", "1" : { "1" : null, "new" : false }, "_i" : [ "1" ], "alt" : "A", "end" : 232449, "ref" : "G" }
{ "_id" : " 1:    233473:C:G", "1" : { "1" : null, "new" : false }, "_i" : [ "1" ], "alt" : "G", "end" : 233473, "ref" : "C" }
{ "_id" : " 1:    233476:G:A", "1" : { "1" : null, "new" : false }, "_i" : [ "1" ], "alt" : "A", "end" : 233476, "ref" : "G" }
{ "_id" : " 1:    233515:A:G", "1" : { "1" : null, "new" : false }, "_i" : [ "1" ], "alt" : "G", "end" : 233515, "ref" : "A" }

Looking at `studies` shows mainly shows the information from the VCF header 

(omitted for brevity, but can be accessed with:
```
!docker exec -it opencga-demo_example_data mongo opencga-demo_testuser_testproj --eval "db.studies.find()"
```
)

`tasks` seem to be a log of operations done to the database. Since we have only done one operations, we can only assume that `directLoad` refers to the data ingestion.

In [25]:
%%time
!docker exec -it opencga-demo_example_data mongo opencga-demo_testuser_testproj --eval "db.tasks.find()"

MongoDB shell version v4.0.5
connecting to: mongodb://127.0.0.1:27017/opencga-demo_testuser_testproj?gssapiServiceName=mongodb
Implicit session: session { "id" : UUID("7ea1b285-0d82-4f51-b513-8807eb817c6a") }
MongoDB server version: 4.0.5
{ "_id" : "1_1", "fileIds" : [ 1 ], "id" : 1, "name" : "storage.mongodb.directLoad", "operationName" : "storage.mongodb.directLoad", "status" : { "2025-03-10T13:52:07&#46;059+0000" : "RUNNING", "2025-03-10T13:58:46&#46;201+0000" : "DONE", "2025-03-10T13:58:51&#46;150+0000" : "READY" }, "studyId" : 1, "timestamp" : NumberLong("1741614727058"), "type" : "LOAD", "_lock" : { "write" : null } }

What's next:
    Try Docker Debug for seamless, persistent debugging tools in any container or image → docker debug opencga-demo_example_data
    Learn more at https://docs.docker.com/go/debug-cli/
CPU times: user 17.3 ms, sys: 22.4 ms, total: 39.7 ms
Wall time: 670 ms


Finally, `variants` contain the the variant data, i.e. the data from each row of the original VCF. (Here showing the first five from the collection)

In [28]:
%%time
!docker exec -it opencga-demo_example_data mongo opencga-demo_testuser_testproj --eval "db.variants.find().limit(5)"

MongoDB shell version v4.0.5
connecting to: mongodb://127.0.0.1:27017/opencga-demo_testuser_testproj?gssapiServiceName=mongodb
Implicit session: session { "id" : UUID("abc9a9c8-4e6e-489a-b0d7-cc3173927a1d") }
MongoDB server version: 4.0.5
{ "_id" : " 1:    173052:C:A", "_at" : { "chunkIds" : [ "1_173_1k", "1_17_10k" ] }, "_index" : { "ts" : NumberLong("1741614727228") }, "_r" : [ 1 ], "alternate" : "A", "annotation" : [ ], "chromosome" : "1", "end" : 173052, "ids" : [ "rs564576411" ], "length" : 1, "reference" : "C", "start" : 173052, "studies" : [ { "sid" : 1, "files" : [ { "fid" : 1, "attrs" : { "AA" : ".|||", "AC" : 37, "SAS_AF" : 0.0941, "FILTER" : "PASS", "AF" : 0.10623, "NS" : 2504, "QUAL" : 100, "DP" : 20868, "AN" : 400, "AMR_AF" : 0.0692, "EUR_AF" : 0.0726, "VCF_ID" : "rs564576411", "EAS_AF" : 0.2758, "AFR_AF" : 0.031, "VT" : "SNP" }, "sampleData" : {  } } ], "gt" : { "1|0" : [ 21, 34, 43, 65, 103, 121, 133, 134, 135, 137, 183, 186, 189, 192, 193, 195, 197, 199 ], "0|1" : [ 5, 

Counting them with a MongoDB command, we see that we get the same number as reported by the logs produced by the variant_index job:

In [29]:
%%time
!docker exec -it opencga-demo_example_data mongo opencga-demo_testuser_testproj --eval "db.variants.find().count()"

MongoDB shell version v4.0.5
connecting to: mongodb://127.0.0.1:27017/opencga-demo_testuser_testproj?gssapiServiceName=mongodb
Implicit session: session { "id" : UUID("c920acfb-d482-49d1-8559-24688ced228b") }
MongoDB server version: 4.0.5
1346514

What's next:
    Try Docker Debug for seamless, persistent debugging tools in any container or image → docker debug opencga-demo_example_data
    Learn more at https://docs.docker.com/go/debug-cli/
CPU times: user 15.3 ms, sys: 22.4 ms, total: 37.7 ms
Wall time: 653 ms


To query the database, the `opencga.sh variant query` command can be used. It should technicaly be possible to make the queries directly to the MongoDB as well, assuming that we can figure out the database schema. But for now, let's limit our investigation to the OpenCGA CLI. A list of all the supported queries can be found by:

In [34]:
%%time
!docker exec -it opencga-demo_example_data ./opencga.sh variant query --help


Usage:   opencga.sh variant query [options]

Options:
                          --alternate	STRING	Main alternate allele 
                  --annotation-exists	BOOLEAN	Return only annotated variants 
                  --approximate-count	BOOLEAN	Get an approximate count, instead of an exact total count. Reduces execution
                                              time
    --approximate-count-sampling-size	INT	Sampling size to get the approximate count. Larger values increase accuracy but
                                              also increase execution time
                            --biotype	STRING	List of biotypes, e.g. protein_coding 
                           --clinical	STRING	Clinical source: clinvar, cosmic 
          --clinical-confirmed-status	BOOLEAN	Clinical confirmed status 
              --clinical-significance	STRING	Clinical significance: benign, likely_benign, likely_pathogenic, pathogenic 
                             --cohort	STRING	Select variants with calc

## 4.2. Subsetting on a genomic range

Querying on a region is done using the `--region` flag. We also need to specify which study we want to query. (We need to be logged in with the testuser account to be able to access the study.)

If we query for a range that only includes the first variant of the VCF, we expect to get one result back:

In [35]:
%%time
!docker exec -it opencga-demo_example_data ./opencga.sh variant query --study teststudy --region 1:1-10200

---
- time: 6
  events: []
  numResults: 1
  results:
  - chromosome: "1"
    alternate: "C"
    studies:
    - stats: []
      samples: []
      files: []
      studyId: "testuser@testproj:teststudy"
      issues: []
      secondaryAlternates: []
      scores: []
      sampleDataKeys: []
    names:
    - "rs367896724"
    reference: ""
    type: "INDEL"
    id: "1:10178:-:C"
    annotation:
      chromosome: "1"
      start: 10178
      end: 10177
      reference: ""
      alternate: "C"
      additionalAttributes:
        opencga:
          attribute:
            release: "1"
    start: 10178
    end: 10177
    strand: "+"
    length: 1
  resultType: ""
  numMatches: -1
  numInserted: 0
  numUpdated: 0
  numDeleted: 0
  numErrors: 0
  attributes:
    numSamples: 0
    numTotalSamples: 0
    source: "mongodb"
  numTotalResults: -1
  federationNode:
    id: "primary"
    uri: "http://localhost:9090/opencga/webservices/rest/"
    commit: "90819ce78287113e7f9333b87de877288051e5ec"
    ve

In this version of OpenCGA, text format results are not supported, so we get the results in YAML format. One result was returned, as was expected. We can notice that we did not get all the INFO column values, and, more importantly, did not get any of the genotype calls. It seems like we also need to specify which samples we want data from. Let's do that next.


## 4.3. Subsetting on sample names


If we extend the previous query with the `--sample` flag, we will get the INFO column values and the genotypes. Let's ask for data from three specific samples:

In [67]:
%%time
!docker exec -it opencga-demo_example_data ./opencga.sh variant query --study teststudy --region 1:1-10200 --sample HG00096,HG00097,HG00099

---
- time: 12
  events: []
  numResults: 1
  results:
  - names:
    - "rs367896724"
    type: "INDEL"
    reference: ""
    id: "1:10178:-:C"
    annotation:
      chromosome: "1"
      start: 10178
      end: 10177
      reference: ""
      alternate: "C"
      additionalAttributes:
        opencga:
          attribute:
            release: "1"
    studies:
    - stats: []
      files:
      - fileId: "1kG_p3_chr1_first_200_samples_c1.vcf.gz"
        call:
          variantId: "1:10177:A:AC"
          alleleIndex: 0
        data:
          AA: "|||unknown(NO_COVERAGE)"
          AC: "116"
          SAS_AF: "0.4949"
          FILTER: "PASS"
          AF: "0.425319"
          NS: "2504"
          QUAL: "100.0"
          DP: "103152"
          AN: "400"
          AMR_AF: "0.3602"
          EUR_AF: "0.4056"
          VCF_ID: "rs367896724"
          EAS_AF: "0.3363"
          AFR_AF: "0.4909"
          VT: "INDEL"
      samples:
      - fileIndex: 0
        data:
        - "1|0"
      - 

In `yq` syntax, the INFO colum values are located in `.results[0].studies[0].files[0].data` and the genotype calls are in `.results[0].studies[0].samples[].data`. The name and order of the samples are not listed in the result but are of course in the query. 

The results show that `HG00096,HG00097,HG00099` have `1|0, 0|1, 0|1` phased genotypes, respectively. A sanity-check with the original VCF file indeed confirms this:

In [69]:
!gzcat ./input_data_temp/1kG_p3_chr1_first_200_samples_c1.vcf.gz | awk '!/^#/ {print; count++; if (count==1) exit}'

1	10177	rs367896724	A	AC	100	PASS	AC=116;AF=0.425319;AN=400;NS=2504;DP=103152;EAS_AF=0.3363;AMR_AF=0.3602;AFR_AF=0.4909;EUR_AF=0.4056;SAS_AF=0.4949;AA=|||unknown(NO_COVERAGE);VT=INDEL	GT	1|0	0|1	0|1	1|0	0|0	1|0	1|0	1|0	1|0	0|0	0|0	0|0	0|0	0|0	0|0	0|0	0|1	1|0	0|0	0|0	1|0	0|0	0|0	0|0	0|1	1|0	0|1	0|1	0|1	0|1	1|0	0|0	1|0	1|0	0|0	0|1	0|0	0|0	1|0	0|1	1|0	0|0	1|0	1|0	0|0	1|0	0|1	0|1	0|0	0|0	1|0	1|0	0|0	0|0	0|1	0|0	0|0	1|0	1|1	1|0	0|1	0|0	0|0	1|1	0|1	0|0	0|1	0|1	0|0	1|0	1|0	1|0	0|1	0|0	1|0	1|0	1|0	0|0	1|0	0|0	0|1	0|1	1|0	0|1	1|1	0|0	0|1	0|0	1|0	0|0	0|0	1|0	0|0	0|0	0|0	1|0	1|0	0|0	0|1	0|0	1|0	0|0	1|0	0|1	1|0	0|1	0|1	0|1	1|0	1|0	0|0	0|0	0|0	0|0	0|0	1|0	0|1	0|0	0|0	0|0	0|1	1|0	1|0	1|0	1|0	1|0	0|0	0|1	0|1	0|0	0|0	0|0	0|0	1|0	0|1	0|0	0|0	0|0	0|1	0|1	1|0	0|0	0|0	0|0	1|0	0|0	1|0	0|0	0|1	0|1	0|0	0|0	0|1	0|0	1|0	0|0	0|1	1|0	0|1	0|0	1|0	1|0	0|0	0|1	1|1	0|0	1|1	0|1	0|0	1|0	1|0	0|1	0|0	0|1	0|0	0|0	0|0	0|0	0|0	0|0	0|1	0|0	0|0	0|0	0|1	0|1	1|0	0|1	0|0	0|0	0|1	1|0	0|0	0|0	1|0	0|0	1|0	0|0	0|0	0|0
gzcat: error 

## 4.4. Subsetting on a variant ID

There is a flag for querying based on variant ID. But it does not seem to accept the variant name as an input? In this example, we have a very short region, but it still returns all variants from the region as opposed to the expected single variant.

In [83]:
%%time
!docker exec -it opencga-demo_example_data ./opencga.sh variant query --study teststudy --region 1:1-14000 --sample HG00096,HG00097,HG00099 --id rs367896724

---
- time: 7
  events: []
  numResults: 6
  results:
  - chromosome: "1"
    alternate: "C"
    studies:
    - stats: []
      samples:
      - fileIndex: 0
        data:
        - "1|0"
      - fileIndex: 0
        data:
        - "0|1"
      - fileIndex: 0
        data:
        - "0|1"
      files:
      - fileId: "1kG_p3_chr1_first_200_samples_c1.vcf.gz"
        call:
          variantId: "1:10177:A:AC"
          alleleIndex: 0
        data:
          AA: "|||unknown(NO_COVERAGE)"
          AC: "116"
          SAS_AF: "0.4949"
          FILTER: "PASS"
          AF: "0.425319"
          NS: "2504"
          QUAL: "100.0"
          DP: "103152"
          AN: "400"
          AMR_AF: "0.3602"
          EUR_AF: "0.4056"
          VCF_ID: "rs367896724"
          EAS_AF: "0.3363"
          AFR_AF: "0.4909"
          VT: "INDEL"
      scores: []
      secondaryAlternates: []
      studyId: "testuser@testproj:teststudy"
      sampleDataKeys:
      - "GT"
      issues: []
    names:
    - "r

The docstring for `--id` seemed like it would fit this query, but obviously something did not work.

```
--id	STRING	List of IDs, these can be rs IDs (dbSNP) or variants in the format
            chrom:start:ref:alt, e.g. rs116600158,19:7177679:C:T
```

We will leave this as is for now and continue with other queries.

## 4.5. Subsetting based on variables from the INFO column of the VCF

From the docstrings, it looks like the `--sample-data` flag would be sufficient.


```
--sample-data	STRING	Filter by any SampleData field from samples. [{sample}:]{key}{op}{value}[,;]* .
                    If no sample is specified, will use all samples from 'sample' or 'genotype'
                    filter. e.g. DP>200 or HG0097:DP>200,HG0098:DP<10 . Many FORMAT fields can be
                    combined. e.g. HG0097:DP>200;GT=1/1,0/1,HG0098:DP<10
```

The question is about how to format the queries. Let's say that we want to display all variants for this range and samples that have an allele frequency value >0.5. That should return two out of the three variants for this range and samples. Howeer, the following command returns nothing, not even an error message:

In [86]:
%%time
!docker exec -it opencga-demo_example_data ./opencga.sh variant query --study teststudy --region 1:1-12000 --sample HG00096,HG00097,HG00099 --sample-data AF>0.5

CPU times: user 69.2 ms, sys: 32.3 ms, total: 101 ms
Wall time: 3.02 s


Looking back at previous results, it seemed like the INFO column values were formatted as strings, e.g. `AF: "0.425319"`. What happens if we format the query as a string instead?

In [87]:
%%time
!docker exec -it opencga-demo_example_data ./opencga.sh variant query --study teststudy --region 1:1-12000 --sample HG00096,HG00097,HG00099 --sample-data "AF>0.5"

ERROR: Execution error : Got server error 'Malformed "sampleData" query : "AF>0.5". FORMAT field "AF" not found. Available keys in study: [GT]'

What's next:
    Try Docker Debug for seamless, persistent debugging tools in any container or image → docker debug opencga-demo_example_data
    Learn more at https://docs.docker.com/go/debug-cli/
CPU times: user 69.2 ms, sys: 34.3 ms, total: 104 ms
Wall time: 3.09 s


Hmm, an error message. So it seems like `--sample-data` does not act on the INFO column values from the VCF but on the FORMAT column? Is the problem that these values can technically go in the FORMAT column, but does not in our example data? 

Since there is a clear discrpency between the docstring, the previous query YAML outputs, and this error message, it seems like this query operation does not work for the example data. We will leave it like that for now, but there is clearly more things to be investigated here.

## 4.6 Subset on variant type (e.g. all INDELs)

We can specify the variant type using the `--type` flag. For example, all INDELs in this range and list of samples:

In [82]:
%%time
!docker exec -it opencga-demo_example_data ./opencga.sh variant query --study teststudy --region 1:1-14000 --sample HG00096,HG00097,HG00099 --type INDEL

---
- time: 10
  events: []
  numResults: 3
  results:
  - chromosome: "1"
    alternate: "C"
    names:
    - "rs367896724"
    reference: ""
    type: "INDEL"
    id: "1:10178:-:C"
    annotation:
      chromosome: "1"
      start: 10178
      end: 10177
      reference: ""
      alternate: "C"
      additionalAttributes:
        opencga:
          attribute:
            release: "1"
    studies:
    - stats: []
      samples:
      - fileIndex: 0
        data:
        - "1|0"
      - fileIndex: 0
        data:
        - "0|1"
      - fileIndex: 0
        data:
        - "0|1"
      files:
      - fileId: "1kG_p3_chr1_first_200_samples_c1.vcf.gz"
        call:
          variantId: "1:10177:A:AC"
          alleleIndex: 0
        data:
          AA: "|||unknown(NO_COVERAGE)"
          AC: "116"
          SAS_AF: "0.4949"
          FILTER: "PASS"
          AF: "0.425319"
          NS: "2504"
          QUAL: "100.0"
          DP: "103152"
          AN: "400"
          AMR_AF: "0.3602"
  

In section 4.4., the same query without the `--type` parameters returned six variants. Adding the filter for INDELS reduced this to three variants, omitting some SNVs. Good.

# 5. Export the dataset from TileDB array back to VCF

The CLI has fuction to export the data. In this version of OpenCGA, it is `./opencga.sh variant export-run`.  It can be run with:

In [65]:
%%time
!docker exec -it opencga-demo_example_data ./opencga.sh variant export-run --study teststudy --outdir export_dir --output-file-name test_output.txt

#ID	TOOL_ID	SUBMISSION_DATE	STATUS	EVENTS	START	RUNNING_TIME	INPUT	OUTPUT
variant-export.20250314134022.j5YZPQ	variant-export	2025-03-14 13:40:22	PENDING	-	-	-	-	-

What's next:
    Try Docker Debug for seamless, persistent debugging tools in any container or image → docker debug opencga-demo_example_data
    Learn more at https://docs.docker.com/go/debug-cli/
CPU times: user 87 ms, sys: 37.4 ms, total: 124 ms
Wall time: 3.75 s


However, this job aborted with the following error message:
```
INFO  ProgressLogger:164 - Export variants 572270/1346514 42.5% up to position 1:97156678:A:G
ERROR ParallelTaskRunner:905 - Error writing batch 57513
java.lang.IllegalArgumentException: Duplicate allele added to VariantContext: N
```

Does this mean that is has issues handling the `N` (any) nucleotide character? Or is it a matter of not being able to handle variants that overlap by flagging them as duplicate alleles? Either which way, this seem like a difficult error to fix without going directly to the MongoDB database and removing the variant that triggered the error.

It would be interesting to try another dataset and see if it runs in the same error message. Since there were issues exporting data with VCF Zarr as well (albeit a different error), we could also start to ask ourselves if there are characteristics in the example dataset that cause this to happen.

# 6. Conclusions from this notebook

The design principles of OpenCGA that focuses on data and user management set it aside from the storage-focused design of TileDB-VCF and VCF Zarr. The tests in this notebook show good promise when it comes to data governance, but small quirks, poor documentation, and the fact that the only version that we were able to get running was an old version put a shade on the overall impression. The lack of documentation make it difficult to know if it is the software or the user that caused the errors.

One can only assume that many of the quirks that were encountered here have been addressed in the four years of GitHub releases for this software that has been made since v2.2.1. For instance: there is promise of a text based (tabular?) output for query results. The YAML output we encountered in this notebbok is not the most human-readable format for genetic variant data... 

The main drawbacks experienced with OpenCGA v2.2.1 are:
- the poor documentation
- the challenging-to-understand installation (in all, highly related to the former)
- the failed export job

A future Part II of this notebook would probably look into the possibilities of getting the latest version of OpenCGA to work. If that can be achieved, many of the small questions and issues in this notebook could be reinvestigated so that we can learn what the current state of this software is.